# DistillRouter — Oracle & Teacher Pipeline


In [1]:
import os, sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists())
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

## 1. Oracle Labeling — Ground Truth

For each question, three candidate models are tried in cost order —
cheapest first — and the question is scored against each one's real
answer. The label is the **cheapest tier that actually answered
correctly**; if none did, it defaults to `large` (the best of the three
tried, even though it failed).

| Tier | Model |
|---|---|
| small | `google/gemma-3-270m-it` |
| medium | `Qwen/Qwen2.5-0.5B-Instruct` |
| large | `google/gemma-3-1b-it` |

### Complete results — both datasets, both oracle-labeled splits

In [2]:
import json, collections

for ds in ["gsm8k", "math"]:
    for split in ["calibration", "train"]:
        counts = collections.Counter()
        for line in open(f"data/oracle/{ds}/{split}.labels.jsonl"):
            counts[json.loads(line)["routing_label"]] += 1
        n = sum(counts.values())
        row = "  ".join(f"{t}={counts[t]:3d} ({counts[t]/n:.1%})" for t in ["small", "medium", "large"])
        print(f"{ds}/{split:11s} n={n:3d}   {row}")

gsm8k/calibration n= 20   small=  0 (0.0%)  medium=  6 (30.0%)  large= 14 (70.0%)
gsm8k/train       n=200   small= 13 (6.5%)  medium=104 (52.0%)  large= 83 (41.5%)
math/calibration n= 20   small=  1 (5.0%)  medium=  5 (25.0%)  large= 14 (70.0%)
math/train       n=200   small=  5 (2.5%)  medium= 39 (19.5%)  large=156 (78.0%)


## 2. Teacher Supervision Generation Workflow

The objective of the teacher is to generate routing supervision for
future student training. The teacher analyzes only the input question
and predicts the minimum model capability required to solve it reliably.
**It never executes downstream models and never observes ground-truth
routing labels for the question it's actually being asked about.**

### Step 1 — Build the prompt



In [3]:
from teacher.fewshot import load_fewshot_examples

fewshot = load_fewshot_examples()
print(f"{len(fewshot)} few-shot examples currently in use:\n")
for example, label in fewshot:
    print(f"  [{label.routing_label:6s}] ({example.dataset}) {example.query[:70]}...")

/home/grbdcn/poorna/DistillRouter/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5 few-shot examples currently in use:

  [small ] (math) What is the degree of the polynomial $(4 +5x^3 +100 +2\pi x^4 + \sqrt{...
  [medium] (gsm8k) Lydia is planning a road trip with her family and is trying to plan a ...
  [medium] (gsm8k) Jack is ordering custom baseball caps for him and his two best friends...
  [large ] (gsm8k) Alton owns a business. He is currently renting a space that costs $20 ...
  [large ] (gsm8k) If Layla scored 104 goals in four hockey games and Kristin scored 24 f...


### Step 3 — Teacher prediction

```json
{"teacher_route": "medium", "routing_reasoning": "Requires multiple dependent reasoning steps while maintaining intermediate values."}
```

Two variants exist, differing only in whether the second field is
requested at all:

In [4]:
from teacher.base import available_teachers
print(available_teachers())

['qwen2.5-3b-v1', 'qwen2.5-3b-v2']




### Step 4 — Validate the output

`teacher_route` must be one of `small`/`medium`/`large`; if parsing
fails or the value is invalid, it defaults to `large` — the same
conservative fallback oracle labeling itself uses, minimizing the risk
of sending a question to a tier too weak for it. `routing_reasoning`
(when requested) is stored exactly as generated — **not verified against
anything** — its only job is capturing the teacher's reasoning process
for later distillation, not being correct.

### Step 5 — Record metadata

```json
{
  "query": "...",
  "dataset": "gsm8k",
  "teacher_route": "medium",
  "routing_reasoning": "Requires multiple dependent reasoning steps while maintaining intermediate values.",
  "input_tokens": 152,
  "output_tokens": 31,
  "latency_seconds": 0.385
}
```

Token counts and latency are captured for analysis/reporting — they are
**not** part of what a future student would be trained to imitate.

### Step 6 — Store the distillation dataset

```
Question ──► Teacher Router ──┬── teacher_route
                               ├── routing_reasoning
                               ├── latency, token counts
                               └── dataset metadata
                                        │
                                        ▼
                          Distillation training dataset
```

## 3. Design Principles

**1. Blind prediction.** The teacher predicts using only the question
text. It never observes downstream model outputs, correctness labels,
oracle routing labels, or benchmark metadata — mirroring exactly what
information a deployed router would have.

**2. Oracle-calibrated demonstrations.** The 5 few-shot examples are
oracle-verified, giving the teacher a stable, consistent definition of
what each tier means — without ever exposing it to oracle information
about the specific question currently being asked.

**3. Knowledge distillation, not ground truth.** The teacher's output is
*supervision*, not verified fact. A future student is trained to imitate
`teacher_route` and `routing_reasoning` — but whether that imitated
policy is actually *good* is a separate question, checked independently
against real oracle ground truth, never folded into training itself.

In [5]:
from evaluation.oracle_check import teacher_accuracy_vs_oracle
import json

result = teacher_accuracy_vs_oracle(["gsm8k", "math"], "calibration", "v2")
print(json.dumps(result, indent=2))

{
  "accuracy": 0.325,
  "n_matched": 40,
  "n_oracle_labeled": 40,
  "per_tier_accuracy": {
    "small": 0.0,
    "medium": 0.8182,
    "large": 0.1429
  },
  "per_tier_support": {
    "small": 1,
    "medium": 11,
    "large": 28
  }
}

## 5. Current status

- **Oracle**: `calibration` (40 questions) and an exhaustively-labeled
  400-question sample of `train` (200 per dataset) are complete. The
  full 2,000-row `train` set is nowhere near oracle-labeled yet — only a
  400-question sample.
- **Teacher**: both `v1` and `v2` are implemented and verified for real
  (§2, §4). `v2` has real output over `calibration` (40 records); neither
  version has been run over `train` at any real scale yet.
- **Prompt design decision, settled**: keep the 5 oracle-verified
  few-shot examples as-is. A simpler, example-free ("zero-shot") prompt
  was considered and deliberately deferred to a future ablation study
  (comparing teacher agreement / oracle accuracy / token cost across
  zero-shot vs. 5-shot) rather than adopted now.
- **Student**: removed earlier this session; a dual-head design (DHRD-style
  — classification head used at inference, a separate reasoning head
  used only during training and discarded afterward) is specified but
  not yet built.

Nothing above is aspirational — every number and example came from
actually running the code against the data that exists on disk right now.